In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
from sklearn.preprocessing import StandardScaler
from pykrx import stock as pykrx_stock
import os
from dotenv import load_dotenv

load_dotenv()

ECOS_API_KEY = os.getenv("ECOS_API_KEY")
FRED_API_KEY = os.getenv("FRED_API_KEY")

START_DATE = "2021-01-01"
END_DATE   = "2024-12-31"

In [6]:
# 1. 주가 데이터 (yfinance)
# ════════════════════════════════════════════════════════════════
TICKERS = {
    "삼성전자":   "005930.KS",
    "SK하이닉스": "000660.KS",
    "NAVER":      "035420.KS",
    "KODEX200":   "069500.KS",
    "KODEX채권":  "114820.KS",
}

def fetch_stock_data():

    TICKERS_KRX = {
        "삼성전자":  "005930",
        "SK하이닉스": "000660",
        "NAVER":     "035420",
        "KODEX200":  "069500",
        "KODEX채권": "114820",
    }

    start_str = START_DATE.replace("-", "")  
    end_str   = END_DATE.replace("-", "")   

    price_dict  = {}
    volume_dict = {}

    for name, ticker in TICKERS_KRX.items():
        try:
            df = pykrx_stock.get_market_ohlcv_by_date(start_str, end_str, ticker)
            if df.empty:
                print(f" {name}({ticker}) 데이터 없음")
                continue
            price_dict[name]  = df["종가"]
            volume_dict[name] = df["거래량"]
            print(f"  {name}: {len(df)}일 수집")
        except Exception as e:
            print(f" {name} 에러: {e}")

    price_df  = pd.DataFrame(price_dict).ffill().dropna()
    volume_df = pd.DataFrame(volume_dict).ffill().dropna()

    # 이상치 탐지
    returns   = price_df.pct_change()
    outliers  = (returns.abs() > 0.15).sum()
    print(f"\n 이상치 탐지 (±15% 초과):\n{outliers}")
    print(f" 주가 수집 완료: {price_df.shape[0]}일 x {price_df.shape[1]}종목")
    return price_df, volume_df

In [7]:
# 2. 금리 데이터 (한국은행 ECOS API)
# ════════════════════════════════════════════════════════════════
def fetch_ecos_series(stat_code, item_code, start, end, name, freq="D"):
    if freq == "D":
        start_str = pd.Timestamp(start).strftime("%Y%m%d")
        end_str   = pd.Timestamp(end).strftime("%Y%m%d")
        date_fmt  = "%Y%m%d"
    else:  # M
        start_str = pd.Timestamp(start).strftime("%Y%m")
        end_str   = pd.Timestamp(end).strftime("%Y%m")
        date_fmt  = "%Y%m"

    url = (
        f"https://ecos.bok.or.kr/api/StatisticSearch"
        f"/{ECOS_API_KEY}/json/kr/1/1000"
        f"/{stat_code}/{freq}/{start_str}/{end_str}/{item_code}"
    )

    resp = requests.get(url, timeout=10)
    data = resp.json()
    rows = data.get("StatisticSearch", {}).get("row", [])

    if not rows:
        print(f" {name} 데이터 없음 — URL: {url}")
        print(f"  응답: {data}")
        return pd.Series(dtype=float, name=name)

    times  = [r["TIME"] for r in rows]
    values = [float(r["DATA_VALUE"]) for r in rows]

    series = pd.Series(values, name=name)
    series.index = pd.to_datetime(times, format=date_fmt)
    series.index.name = "date"
    return series


def fetch_rate_data():

    base_rate = fetch_ecos_series(
        "722Y001", "0101000", START_DATE, END_DATE, "기준금리", freq="M"
    )
    bond_3y = fetch_ecos_series(
        "817Y002", "010200000", START_DATE, END_DATE, "국고채3년", freq="D"
    )
    
    base_rate.index = pd.DatetimeIndex(base_rate.index)
    bond_3y.index   = pd.DatetimeIndex(bond_3y.index)

    base_rate_daily = base_rate.resample("D").ffill()
    bond_3y_daily   = bond_3y.resample("D").ffill()

    rate_df = pd.concat([base_rate_daily, bond_3y_daily], axis=1).ffill()

    print(f" 금리 수집 완료: {rate_df.shape[0]}일 x {rate_df.shape[1]}항목")
    return rate_df

In [8]:
# 3. VIX 공포지수 (FRED API)
# ════════════════════════════════════════════════════════════════
def fetch_vix_data():

    url = (
        f"https://api.stlouisfed.org/fred/series/observations"
        f"?series_id=VIXCLS"
        f"&observation_start={START_DATE}"
        f"&observation_end={END_DATE}"
        f"&api_key={FRED_API_KEY}"
        f"&file_type=json"
    )
    resp = requests.get(url, timeout=10)
    obs  = resp.json()["observations"]

    vix = pd.Series(
        {o["date"]: float(o["value"]) for o in obs if o["value"] != "."},
        name="VIX"
    )
    vix.index = pd.to_datetime(vix.index)
    vix = vix.ffill()

    print(f" VIX 수집 완료: {len(vix)}일")
    return vix



In [9]:
# 4. 데이터 통합 및 저장
# ════════════════════════════════════════════════════════════════
def build_master_dataset(price_df, volume_df, rate_df, vix_series):

    returns_df = price_df.pct_change().dropna()
    macro_df = pd.concat([rate_df, vix_series], axis=1)
    macro_df = macro_df.reindex(returns_df.index).ffill().bfill()
    master_df = pd.concat([returns_df, macro_df], axis=1)

    print(f"  통합 전 shape: {master_df.shape}, NaN 수: {master_df.isnull().sum().sum()}")

    master_df = master_df.dropna()

    print(f"  통합 완료: {master_df.shape[0]}일 x {master_df.shape[1]}컬럼")

    if len(master_df) == 0:
        print("      master_df가 비어있습니다. 날짜 범위 확인 필요")
        print(f"     returns_df 기간: {returns_df.index[0]} ~ {returns_df.index[-1]}")
        print(f"     macro_df 기간:   {macro_df.index[0]} ~ {macro_df.index[-1]}")
        return master_df

    print(f"  기간: {master_df.index[0].date()} ~ {master_df.index[-1].date()}")
    print(f"  컬럼: {list(master_df.columns)}")
    return master_df


In [10]:
# 5. 데이터 품질 리포트
# ════════════════════════════════════════════════════════════════
def data_quality_report(master_df):
    print("\n 데이터 품질 리포트")
    print("=" * 40)
    print(f"총 거래일수: {len(master_df)}")
    print(f"NaN 개수: {master_df.isnull().sum().sum()}")
    print("\n[수익률 기술통계]")
    stock_cols = list(TICKERS.keys())
    print(master_df[stock_cols].describe().round(4))



In [11]:
if __name__ == "__main__":
    price_df, volume_df = fetch_stock_data()
    rate_df             = fetch_rate_data()
    vix_series          = fetch_vix_data()

    master_df = build_master_dataset(price_df, volume_df, rate_df, vix_series)
    data_quality_report(master_df)

    # 저장
    price_df.to_csv("data/price_raw.csv")
    master_df.to_csv("data/master_dataset.csv")
    print("\n 저장 완료: data/price_raw.csv, data/master_dataset.csv")

  삼성전자: 983일 수집
  SK하이닉스: 983일 수집
  NAVER: 983일 수집
  KODEX200: 983일 수집
  KODEX채권: 983일 수집

 이상치 탐지 (±15% 초과):
삼성전자        0
SK하이닉스      0
NAVER       0
KODEX200    0
KODEX채권     0
dtype: int64
 주가 수집 완료: 983일 x 5종목
 금리 수집 완료: 1461일 x 2항목
 VIX 수집 완료: 1024일
  통합 전 shape: (982, 8), NaN 수: 0
  통합 완료: 982일 x 8컬럼
  기간: 2021-01-05 ~ 2024-12-30
  컬럼: ['삼성전자', 'SK하이닉스', 'NAVER', 'KODEX200', 'KODEX채권', '기준금리', '국고채3년', 'VIX']

 데이터 품질 리포트
총 거래일수: 982
NaN 개수: 0

[수익률 기술통계]
           삼성전자    SK하이닉스     NAVER  KODEX200   KODEX채권
count  982.0000  982.0000  982.0000  982.0000  982.0000
mean    -0.0003    0.0006   -0.0002   -0.0001    0.0001
std      0.0156    0.0245    0.0216    0.0114    0.0016
min     -0.1030   -0.1040   -0.0893   -0.0918   -0.0086
25%     -0.0099   -0.0142   -0.0128   -0.0071   -0.0008
50%     -0.0013    0.0000    0.0000    0.0002    0.0001
75%      0.0074    0.0151    0.0103    0.0068    0.0009
max      0.0721    0.0973    0.0994    0.0468    0.0078

 저장 완료: data/price_raw.csv, 